In [ ]:
import os
import shutil
import multiprocessing

from functools import partial
from models import train_model, QuadraticResidualNet_scaled_layernorm, weighted_l1_loss, load_model_checkpoint, generate_fortran_module

import numpy as np
import torch

from torch.optim.lr_scheduler import LinearLR, ExponentialLR, SequentialLR

from physics import set_parameter_ranges, get_parameter_ranges, display_parameter_ranges
from data import create_droplet_batch, create_training_file, read_training_file



# Set-Up

In [ ]:
# Flags
data_directory = "data/"

training_data_file   = "20260729-sobol-pi_chamber-2way.training_data"
validation_data_file = "20260729-sobol-pi_chamber-2way.validation_data"

# ------------ DATA GENERATION SETTINGS --------------
generate_data = False  # If false, use pre-existing data @ data/{training_data_file}
                       #                                 & data/{validation_data_file}

training_droplet_count   = 1024*256   # MUST be a power of 2 for Sobol-sampling!
validation_droplet_count = 1024*64    # MUST be a power of 2 for Sobol-sampling!

training_data_seed   = 100   # These cannot be the same; otherwise, the training/validation data overlap.
validation_data_seed = 200

data_generation_cpu_count = 4   # Should evenly divide the droplet count

# These ranges will be used both for data generation and model data normalization
parameter_ranges = {
    "radius"            : np.array( [-6.75, -4.50] ),
    "temperature"       : np.array( [282.0, 298.0] ),
    "salt_solute"       : np.array( [-17.89, -17.87] ),
    "air_temperature"   : np.array( [282.0, 298.0] ),
    "relative_humidity" : np.array( [0.98, 1.07] ),
    "rhoa"              : np.array( [0.99, 1.01] ),
    "time"              : np.array( [-1.2, -0.85] )
}

# ------------ MODEL TRAINING SETTINGS --------------
train_model       = False            # If false, use pre-existing model @models/{model_name}.pt
serialize_model   = False            # Whether to generate a Fortran droplet model
model_name        = "flippant-gusto" # change if training a new model; otherwise, will overwrite!
device            = "cpu"            # or cuda
peak_lr           = 10.0**(-3.0)
final_lr          = 1.0e-5*8
alpha             = 256.0            # Relative weighting for radius/temperature loss
number_epochs     = 100              # 17600 for paper training set up
callback_interval = 20               # How many epochs to wait before calculating validation loss. Every 50x this, a model checkpoint will be saved
batch_size        = 1024
warmup_fraction   = 0.04             # Linear LR warmup


# Data Generation

In [ ]:
set_parameter_ranges( parameter_ranges )
display_parameter_ranges( get_parameter_ranges() )

In [ ]:
def _data_worker_wrapper(job_index, data_dir, droplet_count, sobol_seed):
    """
    Wrapper function to generate the unique file path and call 
    create_training_file with the correct arguments for this worker.
    """
    # Set training_data_file to be in "data/" with a suffix for its job id
    temp_file_path = os.path.join(data_dir, f"tmp/temp_training_data_{job_index}.dat")
    
    create_training_file(
        file_name=temp_file_path,
        number_droplets=droplet_count,
        sobol_seed=sobol_seed,
        file_index=job_index
    )
    
    return temp_file_path

In [ ]:
def generate_and_pool_data(jobs, final_training_data_file, training_droplet_count, sobol_seed=0):
    data_dir = "data"
    
    # 1. Create a folder called "data"
    os.makedirs(data_dir, exist_ok=True)
    os.makedirs(data_dir + "/tmp", exist_ok=True)
    
    # 2. Prepare arguments for the multiprocessing pool
    task_args = [
        (job_index, data_dir, training_droplet_count, sobol_seed)
        for job_index in range(jobs)
    ]
    
    # 3. Pool this function over "jobs" jobs
    print(f"Dispatching {jobs} jobs across available CPU cores...")
    with multiprocessing.Pool(processes=jobs) as pool:
        # pool.starmap unpacks the tuples in task_args into the wrapper's arguments
        temp_files = pool.starmap(_data_worker_wrapper, task_args)
        
    # 4. After all are done, join all created files into a single file
    print(f"All jobs complete. Joining {len(temp_files)} files into {final_training_data_file}...")
    
    with open(final_training_data_file, 'wb') as outfile:
        for temp_file in temp_files:
            with open(temp_file, 'rb') as infile:
                shutil.copyfileobj(infile, outfile)
            
            os.remove(temp_file)
            
    print("Done!")



In [ ]:
if __name__ == "__main__" and generate_data:
    # Generate Training Data
    per_file_droplet_count = int( training_droplet_count // data_generation_cpu_count )
    
    generate_and_pool_data(
        jobs=data_generation_cpu_count,
        final_training_data_file=data_directory + training_data_file,
        training_droplet_count=per_file_droplet_count,
        sobol_seed=training_data_seed
    )
    
    # Generate Validation Data
    per_file_droplet_count = int( validation_droplet_count // data_generation_cpu_count )
    
    generate_and_pool_data(
        jobs=data_generation_cpu_count,
        final_training_data_file=data_directory + validation_data_file,
        training_droplet_count=per_file_droplet_count,
        sobol_seed=validation_data_seed
    )

In [ ]:
# Load data
training_ins, training_outs     = read_training_file( data_directory + training_data_file )
validation_ins, validation_outs = read_training_file( data_directory + validation_data_file )

training_droplet_count   = training_ins.shape[0]
validation_droplet_count = validation_ins.shape[0]

print(
    f"Training on {training_droplet_count} droplets.\n"
    f"Validating with {validation_droplet_count} droplets."
)

In [ ]:
# Visual Check of Sobol-Sampling
import matplotlib.pyplot as plt

plt.title( "Sobol Spot-Check")
plt.xlabel( "Air Temperature (K)" )
plt.ylabel( "Relative Humidity (%)" )
plt.scatter( np.log10(training_ins[0:1024, 0]), training_ins[0:1024, -3] )

In [ ]:
# Visual Check of Validation/Training Overlap
import matplotlib.pyplot as plt

plt.title( "Validation/Training Collision Spot-Check")
plt.xlabel( "Air Temperature (K)" )
plt.ylabel( "Relative Humidity (%)" )
plt.scatter( validation_ins[:1024, 3], validation_ins[:1024, -3], label="Validation" )
plt.scatter( training_ins[:1024, 3], training_ins[:1024, -3], label="Training" )
plt.legend()

# Model Training

In [ ]:
batch_count       = int( training_ins.shape[0]//batch_size * number_epochs )
warmup_epochs     = int( number_epochs * warmup_fraction )

model     = QuadraticResidualNet_scaled_layernorm()
criterion = partial( weighted_l1_loss, alpha=alpha )
optimizer = torch.optim.Adam( model.parameters(), lr=peak_lr, weight_decay=0.0, fused=True )

model.to( device )
model.train()

if device == "cpu":
    model = torch.compile( model )

gamma = np.exp(np.log(final_lr / peak_lr) / (number_epochs - warmup_epochs))


In [ ]:
warmup_scheduler = LinearLR(
    optimizer,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=warmup_epochs
)

exponential_scheduler = ExponentialLR(
    optimizer,
    gamma=gamma
)

lr_scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, exponential_scheduler],
    milestones=[warmup_epochs]
)

In [ ]:
training_data = (
    training_ins,
    training_outs
)
validation_data = (
    validation_ins,
    validation_outs
)

In [ ]:
if train_model:
    training_loss, validation_loss = train_model(
        model,
        criterion,
        optimizer,
        device,
        number_epochs,
        training_data,
        validation_data,
        checkpoint_prefix="models/" + model_name,
        epoch_callback=None,
        quadratic_loss_flag=True,
        batch_size=batch_size,
        lr_scheduler=lr_scheduler,
        callback_interval=callback_interval
    )
    
    training_epochs   = np.arange( len( training_loss ) )
    validation_epochs = np.arange( 0, number_epochs, callback_interval )
    
    plt.figure()
    plt.title( "Training and Validation Loss" )
    plt.xlabel( "Epoch" )
    plt.ylabel( "Loss" )
    plt.grid( color="k", alpha=0.2 )
    
    plt.plot( training_epochs, training_loss, label="Training Loss" )
    plt.plot( validation_epochs, validation_loss, label="Validation Loss", marker="o" )
    
    plt.yscale( "log" )
    plt.legend()
    plt.show()
else:
    model, parameter_ranges, training_history, validation_history = load_model_checkpoint( "models/" + model_name + ".pt" )

In [ ]:
# Model spot check

from analysis import plot_droplet_size_temperatures_domain
from physics import generate_random_droplets_sobol, set_parameter_ranges

droplet = generate_random_droplets_sobol( 1, sobol_seed=np.random.randint(0, 1000) )[0]

print( "NOTE: Remember the radius parameter ranges! High RH can lead to model radius exceeding 10^(-4.5)m ~= 316 micron.\n"
       "      In such a case, model performance will degrade. Do not take this as an indication of model performance.\n\n" )

plot_droplet_size_temperatures_domain( droplet, model, dt=0.1, final_time=100.0 )

# Fotran Module Generation

**NOTE:** You must move this to `NTLP/droplet_model.f90` and make for proper serialization! See `README.md` for more information.

In [ ]:
from models import generate_fortran_module

if serialize_model:
    serialization_model = model._orig_mod if device == "cpu" else model
    generate_fortran_module( "models/droplet_model_{:s}.f90".format( model_name ), model_name, serialization_model, parameter_ranges, True, True, True, True )